<h1 align="center">Rwanda Maize–Urea Trial</h1>
<h2 align="center">5. Data Enrichment: Rainfall & Soil</h2>

**Goal:** fill two gaps the brief highlights:
1. **Soil:** `tot_N`, `org_C` and `sand` are completely empty. They are filled from **SoilGrids 2.0** (ISRIC, ~250 m resolution).
2. **Weather:** there is no weather data. The variable `rain_plant_silking`, total rainfall from planting to silking, is added from **CHIRPS v2.0** daily rainfall (~5 km resolution).

Both downloads need internet access, so they live in standalone scripts that cache every response and can be stopped and restarted:

```bash
python scripts/fetch_soilgrids.py         # ~5 requests/min (SoilGrids fair-use limit); cached per location
python scripts/fetch_chirps_rainfall.py   # one small subset download from the IRI Data Library
python run_pipeline.py                    # picks up 01_Data/external/*.csv automatically
```

The first attempt at this (v1 of the project) timed out on the SoilGrids REST endpoint. The new script retries with back-off, respects the rate limit and documents two alternative routes (the WCS service and Google Earth Engine).

> **Data note.** The trial data is confidential and is not in this repository. The outputs shown were produced on the real data. If you run this notebook without it, it uses the synthetic sample in `01_Data/sample/` and the numbers will differ.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "02_Notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Image, display

from maize import config, data

pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 30)
MODE = config.data_mode()
OUT = config.output_paths(MODE)
print("Data source:", "confidential trial data (01_Data/raw)" if MODE == "raw" else "synthetic sample (01_Data/sample)")

Data source: confidential trial data (01_Data/raw)


## 1. What needs to be fetched

In [2]:
src = data.load_sources(MODE)
fields = src["field"]
pd.Series({
    "Field-seasons": len(fields),
    "Unique locations (4 d.p.)": fields[["lat", "long"]].round(4).drop_duplicates().shape[0],
    "Soil variables 100% empty": ", ".join(c for c in ["tot_N", "org_C", "sand"] if src["soil"][c].isna().all()),
    "Plots needing a rainfall window": len(src["plot"]),
    "Plots without a silking date (window undefined)": src["plot"]["silking_date"].isna().sum(),
}, name="").to_frame()

,
Field-seasons,399
Unique locations (4 d.p.),267
Soil variables 100% empty,"tot_N, org_C, sand"
Plots needing a rainfall window,1598
Plots without a silking date (window undefined),9


**Method choices**
* **SoilGrids depth.** Values are averaged over 0–30 cm, weighted by layer thickness (0–5, 5–15, 15–30 cm), which matches the maize rooting zone where fertiliser is applied. They are converted to conventional units: N and SOC in g/kg, sand and clay in %, pH in water.
* **Modelled vs measured.** SoilGrids values are kept in separate `sg_*` columns. The measured pH is never overwritten.
* **Rainfall window.** Rainfall is summed daily from planting to silking, inclusive, at the CHIRPS cell nearest each field. Plots without a silking date, or with dates out of order, are left missing.
* **Brief deliverable 5.** `01_Data/external/Soil_PL_filled.csv` is the soil table for every plot, with `tot_N`, `org_C` and `sand` filled and a source column for each.

## 2. Status of the enrichment files

In [3]:
external = data.load_external()
status = {
    "CHIRPS rainfall (chirps_rain_plant_silking.csv)": "rainfall" in external,
    "SoilGrids soil (soilgrids_soil.csv)": "soilgrids" in external,
    "Filled soil table (Soil_PL_filled.csv)": (config.EXTERNAL_DIR / "Soil_PL_filled.csv").exists(),
}
pd.Series(status, name="available").to_frame()

,available
CHIRPS rainfall (chirps_rain_plant_silking.csv),False
SoilGrids soil (soilgrids_soil.csv),False
Filled soil table (Soil_PL_filled.csv),False


## 3. Explore the added variables

This section runs once the enrichment files exist. Until then it prints a reminder.

In [4]:
_, combined = data.build_combined(MODE)
added = [c for c in combined.columns if c.startswith(("sg_", "rain_"))]
if not added:
    print("No enrichment data yet - run the two scripts above on a machine with internet access.")
else:
    ok = combined[~combined["flag_implausible_yield"]]
    display(ok[added].describe().T)
    display(ok[added + ["yield_tha"]].corr()["yield_tha"].drop("yield_tha").to_frame("correlation with yield"))
    display(ok.groupby("meg_env")[added].mean())

No enrichment data yet - run the two scripts above on a machine with internet access.


## What happens next

Once the files exist, every later step uses them automatically:
* notebook 06 adds `rain_plant_silking` and the `sg_*` variables to the **driver importance** ranking;
* `run_pipeline.py` rewrites the combined dataset and figures;
* the audit log marks AUD-20 and AUD-22 as resolved.

**Next:** notebook 06, yield prediction and drivers.